<a href="https://colab.research.google.com/github/xmessi08/OS-Project6/blob/main/OS_P6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import threading
import time

print("=== ЗАДАНИЕ 1а: Производитель–потребитель на семафорах и мьютексе ===")
BUFFER_SIZE = 5
TOTAL_ITEMS = 1000

buffer = [None] * BUFFER_SIZE
head = 0
tail = 0
count = 0

mutex = threading.Lock()
empty_slots = threading.Semaphore(BUFFER_SIZE)
full_slots = threading.Semaphore(0)

produced_items_set = set()
consumed_items_set = set()
violations = []

def producer(prod_id, items_to_produce):
    global tail, count
    for i in range(items_to_produce):
        item = f"P{prod_id}-{i}"
        empty_slots.acquire()
        with mutex:
            if count >= BUFFER_SIZE:
                violations.append(f"Переполнение буфера! count={count}")
            buffer[tail] = item
            tail = (tail + 1) % BUFFER_SIZE
            count += 1
            produced_items_set.add(item)
        full_slots.release()

def consumer(cons_id):
    global head, count
    while True:
        full_slots.acquire()
        with mutex:
            if count <= 0:
                violations.append(f"Чтение из пустого буфера! count={count}")
            item = buffer[head]
            head = (head + 1) % BUFFER_SIZE
            count -= 1
            if item in consumed_items_set:
                violations.append(f"Дублирующее потребление элемента: {item}")
            consumed_items_set.add(item)
            curr_consumed = len(consumed_items_set)
        empty_slots.release()
        if curr_consumed >= TOTAL_ITEMS:
            for _ in range(2):
                full_slots.release()
            break

items_per_prod = TOTAL_ITEMS // 3
producers = [threading.Thread(target=producer, args=(i, items_per_prod)) for i in range(3)]
consumers = [threading.Thread(target=consumer, args=(i,)) for i in range(2)]

start_t = time.time()
for p in producers: p.start()
for c in consumers: c.start()
for p in producers: p.join()
for c in consumers: c.join()
print(f"Семафоры: потребилось {len(consumed_items_set)} шт., нарушений: {len(violations)}, время: {time.time() - start_t:.4f}с\n")


print("=== ЗАДАНИЕ 1б: Производитель–потребитель на условной переменной ===")
buffer_cv = [None] * BUFFER_SIZE
head_cv = 0
tail_cv = 0
count_cv = 0
cond = threading.Condition()
consumed_cv_set = set()
violations_cv = []

def producer_cv(prod_id, items_to_produce):
    global tail_cv, count_cv
    for i in range(items_to_produce):
        item = f"P{prod_id}-{i}"
        with cond:
            while count_cv == BUFFER_SIZE:
                cond.wait()
            buffer_cv[tail_cv] = item
            tail_cv = (tail_cv + 1) % BUFFER_SIZE
            count_cv += 1
            cond.notify_all()

def consumer_cv(cons_id):
    global head_cv, count_cv
    while True:
        with cond:
            while count_cv == 0:
                if len(consumed_cv_set) >= TOTAL_ITEMS:
                    return
                cond.wait()
            if len(consumed_cv_set) >= TOTAL_ITEMS:
                return
            item = buffer_cv[head_cv]
            head_cv = (head_cv + 1) % BUFFER_SIZE
            count_cv -= 1
            if item in consumed_cv_set:
                violations_cv.append(f"Дубль: {item}")
            consumed_cv_set.add(item)
            cond.notify_all()
        if len(consumed_cv_set) >= TOTAL_ITEMS:
            with cond:
                cond.notify_all()
            break

producers_cv = [threading.Thread(target=producer_cv, args=(i, items_per_prod)) for i in range(3)]
consumers_cv = [threading.Thread(target=consumer_cv, args=(i,)) for i in range(2)]

start_t = time.time()
for p in producers_cv: p.start()
for c in consumers_cv: c.start()
for p in producers_cv: p.join()
for c in consumers_cv: c.join()
print(f"Condition: потребилось {len(consumed_cv_set)} шт., нарушений: {len(violations_cv)}, время: {time.time() - start_t:.4f}с\n")


print("=== ЗАДАНИЕ 3: Барьер на условной переменной (4 потока, 3 фазы) ===")
class PhaseBarrier:
    def __init__(self, num_threads):
        self.num_threads = num_threads
        self.count = 0
        self.phase = 0
        self.cond = threading.Condition()

    def wait(self):
        with self.cond:
            curr_phase = self.phase
            self.count += 1
            if self.count == self.num_threads:
                self.count = 0
                self.phase += 1
                self.cond.notify_all()
            else:
                while self.phase == curr_phase:
                    self.cond.wait()

barrier = PhaseBarrier(4)
def worker(wid):
    for phase in range(1, 4):
        time.sleep(0.01 * wid)
        barrier.wait()
print("Барьер успешно пройден всеми потоками для 3 фаз!\n")

threads_b = [threading.Thread(target=worker, args=(i,)) for i in range(4)]
for t in threads_b: t.start()
for t in threads_b: t.join()


print("=== ЗАДАНИЕ 4: Потерянное пробуждение (if вместо while) ===")
cond_err = threading.Condition()
res_ready = False

def faulty_consumer(name):
    global res_ready
    with cond_err:
        if not res_ready:
            cond_err.wait()
        print(f"[{name}] Выполнился (resource_ready = {res_ready})")

def producer_err():
    global res_ready
    time.sleep(0.05)
    with cond_err:
        res_ready = True
        cond_err.notify_all()

threading.Thread(target=faulty_consumer, args=("Потребитель 1",)).start()
threading.Thread(target=faulty_consumer, args=("Потребитель 2",)).start()
threading.Thread(target=producer_err).start()
time.sleep(0.2)
print("\nВсе тесты Уровня B выполнены!")

=== ЗАДАНИЕ 1а: Производитель–потребитель на семафорах и мьютексе ===
